In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data/raw/customers.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")

customers.info()
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 766 entries, 0 to 765
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  766 non-null    int64
 1   order_id       766 non-null    int64
 2   product_id     766 non-null    int64
 3   quantity       766 non-null    int64
 4   unit_price     766 non-null    int64
dtypes: int64(5)
memory usage: 30.1 KB


In [2]:
orders.info()
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        301 non-null    int64
 1   customer_id     301 non-null    int64
 2   order_date      301 non-null    str  
 3   payment_method  301 non-null    str  
 4   order_status    301 non-null    str  
dtypes: int64(2), str(3)
memory usage: 20.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 6.0 KB


In [3]:
# 원본 구조에 대한 이력(Evidence) 을 만든다.
# raw_date = [customers, order_items,orders,products] >>>리스트로 만들수도 있음

raw_data = {
    "customers":customers, 
    "order_items": order_items,
    "orders": orders,
    "products":products
    }
print(raw_date)

NameError: name 'raw_date' is not defined

In [ ]:
summary_list = []

for name, frame in raw_data.items():
    # print(name)
    info = {
        "dataset:": name,
        "rows": len(frame), # 레코드의 개수 불러오는
        "clumns": frame.shape[1],
        "missing_values" : int(frame.isna().sum().sum()),
        "duplicated_rows":int(frame.duplicated().sum()),
    }

    summary_list.append(info)

In [ ]:
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)

processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)

relationship_checks = validate_relationships(
    processed_data
)

In [ ]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(raw_data, processed_data)

In [ ]:
preprocessing_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,766,5,766,6
2,orders,301,5,301,7
3,products,100,4,100,4


In [ ]:
print(raw_data ["order_items"].columns)
#딕셔너리[데이터 컬럼]
print(processed_data ["order_items"].columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'], dtype='str')
Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


In [ ]:
print(raw_data ["orders"].columns)
#딕셔너리[데이터 컬럼]
print(processed_data ["orders"].columns)

Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')


In [ ]:
print(processed_data ["orders"].head())

   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-06-03           card    completed     2026-06   
1         2           77 2025-08-19      naver_pay    cancelled     2025-08   
2         3          138 2025-12-16  bank_transfer    cancelled     2025-12   
3         4           57 2026-02-26      kakao_pay    cancelled     2026-02   
4         5          125 2026-01-17           card    cancelled     2026-01   

  order_dayofweek  
0       Wednesday  
1         Tuesday  
2         Tuesday  
3        Thursday  
4        Saturday  


In [ ]:
print(processed_data ["orders"]["order_dayofweek"].value_counts())
#데이터 프레임안에 한가지만 보고 싶을 때 ["해당컬럼명 추가"]

order_dayofweek
Friday       50
Monday       47
Sunday       47
Tuesday      44
Saturday     44
Thursday     35
Wednesday    34
Name: count, dtype: int64


키 값매핑

In [ ]:
key_map ={
    "customers" : "customer_id",
    "products" : "product_id",
    "orders" : "order_id",
    "order_items" : "order_item_id",
}

In [ ]:
kp_checks = []

for dataset, key in key_map.items():
    frame = processed_data [dataset]
    missing_count = int(frame[key].isna().sum())
    duplicated_count = int(frame[key].duplicated().sum())
    status = ""
    if missing_count == 0 and duplicated_count == 0:
        status = "PASS"
    else:
        status = "FAIL"
    kp_checks.append(
        {
            "dataset": dataset,
            "key": key,
            "missing_count": missing_count,
            "duplicated": duplicated_count,
            "status": status,

        }
    )


In [ ]:
kp_checks

[{'dataset': 'customers',
  'key': 'customer_id',
  'missing_count': 0,
  'duplicated': 0,
  'status': 'PASS'},
 {'dataset': 'products',
  'key': 'product_id',
  'missing_count': 0,
  'duplicated': 0,
  'status': 'PASS'},
 {'dataset': 'orders',
  'key': 'order_id',
  'missing_count': 0,
  'duplicated': 0,
  'status': 'PASS'},
 {'dataset': 'order_items',
  'key': 'order_item_id',
  'missing_count': 0,
  'duplicated': 1,
  'status': 'FAIL'}]

In [ ]:
order_sales = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],

    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

In [ ]:
order_sales.head()

,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,123.0,2026-06-03,completed,both
1,2,1,87,5,25000,123.0,2026-06-03,completed,both
2,3,1,7,3,142000,123.0,2026-06-03,completed,both
3,4,1,9,3,193000,123.0,2026-06-03,completed,both
4,5,2,72,4,189000,77.0,2025-08-19,cancelled,both


In [ ]:
print("병합 전 행 수 :", len(order_items))
print("병합 후 행 수 :", len(order_sales))

order_sales["_merge"].value_counts(dropna=False)

병합 전 행 수 : 766
병합 후 행 수 : 766


_merge
both          765
left_only       1
right_only      0
Name: count, dtype: int64

In [ ]:
expected_line_total = order_items["quantity"] * order_items["unit_price"]
expected_line_total

0      306000
1      125000
2      426000
3      579000
4      756000
        ...  
761    378000
762    700000
763    160000
764    160000
765    700000
Length: 766, dtype: int64

In [ ]:
expected_line_total

0      306000
1      125000
2      426000
3      579000
4      756000
        ...  
761    378000
762    700000
763    160000
764    160000
765    700000
Length: 766, dtype: int64

In [ ]:
#검증해야 될 것

if "line_total" not in order_items.columns:
    order_items["line_total"] = expected_line_total


order_items.info()


<class 'pandas.DataFrame'>
RangeIndex: 766 entries, 0 to 765
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  766 non-null    int64
 1   order_id       766 non-null    int64
 2   product_id     766 non-null    int64
 3   quantity       766 non-null    int64
 4   unit_price     766 non-null    int64
 5   line_total     766 non-null    int64
dtypes: int64(6)
memory usage: 36.0 KB


In [ ]:
completed_order_sales  = order_sales.loc[
    order_sales["order_status"].eq("completed") #등가비교? 함수 .eq
]

print(completed_order_sales.head())
print(completed_order_sales.head()["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0               1         1         100         3      102000        123.0   
1               2         1          87         5       25000        123.0   
2               3         1           7         3      142000        123.0   
3               4         1           9         3      193000        123.0   
12             13         6          83         3       24000         87.0   

    order_date order_status _merge  
0   2026-06-03    completed   both  
1   2026-06-03    completed   both  
2   2026-06-03    completed   both  
3   2026-06-03    completed   both  
12  2026-04-17    completed   both  
order_status
completed    5
Name: count, dtype: int64


In [ ]:
print("전체 주문 건수 :", len(order_sales))
print("전체 주문 건수(completed) :", len(completed_order_sales))
print("전체 주문 건수(completde 외) :", len(order_sales) - len(completed_order_sales))

전체 주문 건수 : 766
전체 주문 건수(completed) : 475
전체 주문 건수(completde 외) : 291


In [ ]:
order_sales["order_status"].value_counts()

order_status
completed    475
cancelled    162
refunded     128
Name: count, dtype: int64

In [ ]:
# 1. groupby를 수행하기 전, 병합 데이터프레임(completed_order_sales)에 매출액 컬럼을 확실하게 추가해 줍니다.
completed_order_sales["line_total"] = completed_order_sales["quantity"] * completed_order_sales["unit_price"]

# 2. 그룹화 및 데이터 집계를 수행합니다.
order_status_sales = (
    completed_order_sales
    .groupby("order_status", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    # 3. "sum" 문자열을 지우고, 매출액이 높은 순(내림차순)으로 올바르게 정렬합니다.
    .sort_values("total_sales", ascending=False)
)

# 4. 결과 출력해서 눈으로 확인하기
order_status_sales

,order_status,total_quantity,total_sales
0,completed,1446,149690000


In [5]:
print(order_status_sales.columns)
print(products.head())

NameError: name 'order_status_sales' is not defined